In [ ]:
run_gridsearch = True
USE_BAYESIAN = False
N_BAYES_TRIALS = 36
skip_best_model_validation = False
skip_best_model_test = False
verbose = False

GPU_SETTING = -1
NUM_ENSEMBLES = 1
NUM_ENSEMBLES_FINAL = 1  # ensemble count for final training (validation/test)
BASIN = "warm_springs"
MODE = "daily"
RUN_LABEL = "CVTEST_V2"
READ_STAMP = "20250815T000000Z"

HYPERPARAM_ENSEMBLE = False
BOOTSTRAP_MODELS = False

# --- Hyperparameter Selection Method ---
use_cv_for_selection = False  # False = single-split scoring, True = CV-averaged
CV_INTERVAL_LENGTH = 2
CV_VALIDATION_LENGTH = 1
CV_INTERVAL_MONTH = 'October'


In [ ]:
hyperparam_space = {
    "hidden_size": [64, 128, 256],
    "output_dropout": [0.4],
    "seq_length": [90],
    "num_layers": [1],
    "epochs": [300],  # was [100] - plateau ES handles stopping
    "batch_size": [64],
    # "schedule_pairs": [  # plateau ES handles LR - no manual schedule needed
    #     ((0.5, 0.25), (0.01, 0.005, 0.001))
    # ]
}

In [ ]:
import sys
import pandas as pd
import os
import itertools
from pathlib import Path
from tqdm import tqdm
import warnings
from datetime import datetime
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
current_dir = os.getcwd()
print(current_dir)

In [ ]:
library_path = os.path.join('..', '..', '..','..','UCB-USACE-RR-PROJECT')
sys.path.insert(0, library_path)
print(sys.path)

In [ ]:
from neuralhydrology.evaluation.metrics import *
from UCB_training.UCB_train import UCB_trainer
from UCB_training.UCB_utils import (fractional_multi_lr, write_paths, to_path_or_list, ensure_output_tree, set_active_context, data_dir, repo_root, get_output_dir, make_run_stamp, get_yaml_path, ctx_for, hparams_exists, save_hparams, load_hparams, runs_latest_path, archive_runs_json, read_csv_artifact, ensure_shared_tree)
from UCB_training.UCB_plotting import (plot_timeseries_comparison, scatter_triptych_pngs_v3, ts_triptych_v3)

In [ ]:
current_path = os.getcwd()
library_path = current_path.split('UCB-USACE-RR-PROJECT')[0] + 'UCB-USACE-RR-PROJECT'

In [ ]:
RUNS_FILE = str(runs_latest_path(BASIN, MODE, RUN_LABEL))
SHOULD_STAMP = run_gridsearch or not (skip_best_model_validation and skip_best_model_test)
RUN_STAMP = make_run_stamp() if SHOULD_STAMP else None
ACTIVE_STAMP = RUN_STAMP if RUN_STAMP is not None else READ_STAMP

In [ ]:
set_active_context(basin=BASIN, resolution=MODE, run_stamp=ACTIVE_STAMP, run_tag=RUN_LABEL, append_stamp_to_filenames=False)
SHARED_FOLDER = ensure_shared_tree(BASIN, MODE)                             
RUNS_PARENT = SHARED_FOLDER / "runs" / (f"{RUN_LABEL}_{RUN_STAMP}" if RUN_STAMP else RUN_LABEL)

print("NH runs will be written under:")
print(RUNS_PARENT.resolve())

In [ ]:
path_to_csv = data_dir()
path_to_yaml = get_yaml_path("warm_springs_dam_nlayer")
path_to_physics_data = path_to_csv / "WarmSprings_Inflow_daily_shift.csv"

In [ ]:
features_with_physics = [
    #from daily.csv
    "DRY CREEK 20 PRECIP-INC SCREENED",
    "DRY CREEK 20 ET-POTENTIAL RUN:BASIN AVERAGE 60 YR",
    "DRY CREEK 30 PRECIP-INC SCREENED",
    "DRY CREEK 30 ET-POTENTIAL RUN:BASIN AVERAGE 60 YR",
    "UKIAH CA HUMIDITY USAF-NOAA",
    "UKIAH CA SOLAR RADIATION USAF-NOAA",
    "UKIAH CA TEMPERATURE USAF-NOAA",
    "UKIAH CA WINDSPEED USAF-NOAA",
    "SANTA ROSA CA HUMIDITY USAF-NOAA",
    "SANTA ROSA CA SOLAR RADIATION USAF-NOAA",
    "SANTA ROSA CA TEMPERATURE USAF-NOAA",
    "SANTA ROSA CA WINDSPEED USAF-NOAA",
    #from Warm_Spring_Inflow.csv
    'Dry Creek 20 ET-POTENTIAL',
    'Dry Creek 20 FLOW',
    'Dry Creek 20 FLOW-BASE',
    'Dry Creek 20 INFILTRATION',
    'Dry Creek 20 PERC-SOIL',
    'Dry Creek 20 SATURATION FRACTION',
    'Dry Creek 30 ET-POTENTIAL',
    'Dry Creek 30 FLOW',
    'Dry Creek 30 FLOW-BASE',
    'Dry Creek 30 INFILTRATION',
    'Dry Creek 30 PERC-SOIL',
    'Dry Creek 30 SATURATION FRACTION',
    'Warm Springs Dam Inflow FLOW',
]

In [ ]:
no_physics_results = []
physics_results = []

In [ ]:
start_time = datetime.utcnow()
print("Start time:", start_time.strftime("%Y-%m-%d %H:%M:%S"))

In [ ]:
import os
import sys
import torch
from pathlib import Path
import multiprocessing as mp
import itertools
import pandas as pd
from tqdm import tqdm
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import warnings
from UCB_training.grid_search_workers import (run_single_experiment_nophysics, run_single_experiment_physics)

# --- set single-threading for all libraries to avoid oversubscription in multiprocessing ---
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1"
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
PROJECT_ROOT = Path.cwd().parents[3]
sys.path.insert(0, str(PROJECT_ROOT))
mp.set_start_method("spawn", force=True)
warnings.filterwarnings("ignore", category=UserWarning)
# --- end ---

# --- determine how many workers to parallelize cross validation, then trials in priority ---
num_cores = os.cpu_count() - 1
def get_pool():
    global GLOBAL_POOL
    if GLOBAL_POOL is None:
        GLOBAL_POOL = mp.Pool(processes=num_cores)
    return GLOBAL_POOL
GLOBAL_POOL = mp.Pool(processes=num_cores)

print("run_gridsearch =", run_gridsearch)
print("hparams_exists =", hparams_exists(BASIN, MODE, RUN_LABEL))

hyperparam_names = list(hyperparam_space.keys())
bayes_log_path = Path(RUNS_PARENT) / f"{RUN_LABEL}_{RUN_STAMP}_bayes_log.csv"
bayes_log_exists = bayes_log_path.exists()


def log_trial(trial_num, value, params):
    import sys
    sys.stdout = sys.__stdout__
    sys.stderr = sys.__stderr__

    print(f"\n[Trial {trial_num}] NSE = {value:.5f}")
    print("Hyperparameters:")
    for k, v in params.items():
        print(f"  {k}: {v}")

if USE_BAYESIAN:

    optuna.logging.set_verbosity(optuna.logging.INFO)

    print(f"\n[BAYESIAN OPTIMIZATION] Using {num_cores} parallel workers\n")

    def load_checkpoint(tag):
        root = _artifact_root(BASIN, MODE)
        arch = root / "hyperparams" / "archive"
        prefix = f"{BASIN}_{MODE}_{RUN_LABEL}"

        if not READ_STAMP:
            print("No READ_STAMP provided - starting fresh.")
            return None, 0

        path = arch / f"{prefix}_{tag}_gridsearch_{READ_STAMP}.csv"

        if path.exists():
            df = pd.read_csv(path)
            completed = len(df)
            print(f"[Bayes] Restored {completed} {tag} trials from {path}")
            return df, completed

        print(f"[Bayes] No archive checkpoint found at {path}")
        return None, 0

    def append_trial_row(df_row: dict, *, basin: str, mode: str, label: str, run_stamp: str, tag: str):
        root = _artifact_root(basin, mode)
        hp_dir = root / "hyperparams"
        arch = hp_dir / "archive"
        prefix = f"{basin}_{mode}_{label}"

        path_latest = hp_dir / f"{prefix}_{tag}_gridsearch.csv"
        path_arch   = arch / f"{prefix}_{tag}_gridsearch_{run_stamp}.csv"

        row = pd.DataFrame([df_row])

        for p in (path_latest, path_arch):
            write_header = not p.exists()
            row.to_csv(p, mode="a", header=write_header, index=False)

    def suggest_from_space(trial):
        comb = []

        for hp, values in hyperparam_space.items():

            # Case 1: list or tuple of complex objects -> categorical
            if isinstance(values, (list, tuple)) and len(values) > 2:
                val = trial.suggest_categorical(hp, values)

            # Case 2: numeric range (low, high)
            elif isinstance(values, tuple) and len(values) == 2:
                lo, hi = values

                # int range
                if isinstance(lo, int) and isinstance(hi, int):
                    val = trial.suggest_int(hp, lo, hi)

                # float range
                else:
                    val = trial.suggest_float(hp, lo, hi)

            # Case 3: list of scalars -> categorical
            else:
                val = trial.suggest_categorical(hp, values)

            comb.append(val)

        return tuple(comb)


    def objective_no_physics(trial):

        comb = suggest_from_space(trial)

        args = (
            trial.number, comb, hyperparam_names, path_to_csv, path_to_yaml,
            GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
            fractional_multi_lr,
            NUM_ENSEMBLES, BOOTSTRAP_MODELS, HYPERPARAM_ENSEMBLE,
            False, False,
            use_cv_for_selection,
            CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH
        )

        try:
            result = run_single_experiment_nophysics(args)
            value = result["NSE"]
        except Exception as e:
            print(f"[Trial {trial.number}] FAILED: {e}")
            return -1e9

        append_trial_row(
            {
                **trial.params,
                "value": value,
                "NSE": value,
            },
            basin=BASIN,
            mode=MODE,
            label=RUN_LABEL,
            run_stamp=RUN_STAMP,
            tag="no_physics",
        )

        log_trial(trial.number, value, trial.params)

        return value


    def objective_physics(trial):

        comb = suggest_from_space(trial)

        args = (
            trial.number, comb, hyperparam_names, path_to_csv, path_to_yaml,
            GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
            fractional_multi_lr,
            NUM_ENSEMBLES, BOOTSTRAP_MODELS, HYPERPARAM_ENSEMBLE,
            features_with_physics,
            path_to_physics_data,
            False, False,
            use_cv_for_selection,
            CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH
        )

        try:
            result = run_single_experiment_physics(args)
            value = result["NSE"]

        except Exception as e:
            print(f"[Trial {trial.number}] FAILED: {e}")
            return -1e9

        append_trial_row(
            {
                **trial.params,
                "value": value,
                "NSE": value,
            },
            basin=BASIN,
            mode=MODE,
            label=RUN_LABEL,
            run_stamp=RUN_STAMP,
            tag="physics",
        )

        log_trial(trial.number, value, trial.params)

        return value


    _, done_no = load_checkpoint("no_physics")
    remaining_no = max(N_BAYES_TRIALS - done_no, 0)

    print(f"[Bayes] No-physics remaining trials: {remaining_no}")

    study_no = optuna.create_study(
        study_name="journal_storage_multiprocess",
        storage=JournalStorage(JournalFileBackend(file_path="./journal.log")),
        load_if_exists=True,
        direction="maximize",
    )

    if remaining_no > 0:
        with mp.Pool(processes=num_cores) as pool:
            study_no.optimize(
                objective_no_physics,
                n_trials=remaining_no,
                n_jobs=1,
                show_progress_bar=True,
            )
    else:
        print("[Bayes] No-physics already complete - skipping.")

    _, done_phys = load_checkpoint("physics")
    remaining_phys = max(N_BAYES_TRIALS - done_phys, 0)

    print(f"[Bayes] Physics remaining trials: {remaining_phys}")

    study_phys = optuna.create_study(
        study_name="journal_storage_multiprocess",
        storage=JournalStorage(JournalFileBackend(file_path="./journal.log")),
        load_if_exists=True,
        direction="maximize",
    )

    if remaining_phys > 0:
        with mp.Pool(processes=num_cores) as pool:
            study_phys.optimize(
                objective_physics,
                n_trials=remaining_phys,
                n_jobs=1,
                show_progress_bar=True,
            )
    else:
        print("[Bayes] Physics already complete - skipping.")

    df_prev_no, _ = load_checkpoint("no_physics")
    df_prev_phys, _ = load_checkpoint("physics")

    df_no_physics = pd.concat([df_prev_no, study_no.trials_dataframe()], ignore_index=True)
    df_physics    = pd.concat([df_prev_phys, study_phys.trials_dataframe()], ignore_index=True)

    df_no_physics.sort_values(by="value", ascending=False, inplace=True)
    df_physics.sort_values(by="value", ascending=False, inplace=True)

    df_no_physics.reset_index(drop=True, inplace=True)
    df_physics.reset_index(drop=True, inplace=True)

    best_no_phys = study_no.best_params
    best_phys = study_phys.best_params

    best_no_phys["model_type"] = "no_physics"
    best_phys["model_type"] = "physics"

    best_params_df = pd.DataFrame([best_no_phys, best_phys])

    save_hparams(
        best_df=best_params_df,
        basin=BASIN,
        mode=MODE,
        label=RUN_LABEL,
        run_stamp=RUN_STAMP,
        df_no=df_no_physics,
        df_phys=df_physics
    )


print("run_gridsearch =", run_gridsearch)
print("hparams_exists =", hparams_exists(BASIN, MODE, RUN_LABEL))

hyperparam_names = list(hyperparam_space.keys())
total_iters = 1
for name in hyperparam_names:
    total_iters *= len(hyperparam_space[name])

elif run_gridsearch or not hparams_exists(BASIN, MODE, RUN_LABEL):
    all_combinations = list(itertools.product(*[hyperparam_space[hp] for hp in hyperparam_names]))
    num_cores = max(1, mp.cpu_count() - 1)
    print(f"\n[GRID SEARCH] Spawning {num_cores} workers\n")

    # NO PHYSICS GRID SEARCH
    task_args_no = [
        (idx, comb, hyperparam_names, path_to_csv, path_to_yaml,
         GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
         fractional_multi_lr, NUM_ENSEMBLES, BOOTSTRAP_MODELS, HYPERPARAM_ENSEMBLE,
         False, False, use_cv_for_selection, CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH,
         None, None, None, None,  # train/val start/end
         None, None, None,  # val_eval_start/end, validation_start_per_frequency
         None, None, None,  # train_ranges, validation_ranges, dataset_name
         0, 1, False)  # fold_id, total_folds, cv_external_queue_mode
        for idx, comb in enumerate(all_combinations)
    ]
    with mp.Pool(processes=num_cores) as pool:
        no_physics_results = list(tqdm(
            pool.imap(run_single_experiment_nophysics, task_args_no),
            total=len(all_combinations), desc="Grid No-Physics", unit="it", ncols=60, ascii=True))
    df_no_physics = pd.DataFrame(no_physics_results)
    df_no_physics.sort_values(by="NSE", ascending=False, inplace=True)
    df_no_physics.reset_index(drop=True, inplace=True)
    print("\n\u2713 No-physics grid complete\n")

    # PHYSICS GRID SEARCH
    task_args_phys = [
        (idx, comb, hyperparam_names, path_to_csv, path_to_yaml,
         GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
         fractional_multi_lr, NUM_ENSEMBLES, BOOTSTRAP_MODELS, HYPERPARAM_ENSEMBLE,
         features_with_physics, path_to_physics_data,
         False, False, use_cv_for_selection, CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH,
         None, None, None, None,  # train/val start/end
         None, None, None,  # val_eval_start/end, validation_start_per_frequency
         None, None, None,  # train_ranges, validation_ranges, dataset_name
         0, 1, False)  # fold_id, total_folds, cv_external_queue_mode
        for idx, comb in enumerate(all_combinations)
    ]
    with mp.Pool(processes=num_cores) as pool:
        physics_results = list(tqdm(
            pool.imap(run_single_experiment_physics, task_args_phys),
            total=len(all_combinations), desc="Grid Physics", unit="it", ncols=60, ascii=True))
    df_physics = pd.DataFrame(physics_results)
    df_physics.sort_values(by="NSE", ascending=False, inplace=True)
    df_physics.reset_index(drop=True, inplace=True)
    print("\n\u2713 Physics grid complete\n")

    # save best params
    best_no_phys = df_no_physics.iloc[0].to_dict()
    best_phys = df_physics.iloc[0].to_dict()
    best_no_phys["model_type"] = "no_physics"
    best_phys["model_type"] = "physics"
    best_params_df = pd.DataFrame([best_no_phys, best_phys])
    save_hparams(best_df=best_params_df, basin=BASIN, mode=MODE, label=RUN_LABEL, run_stamp=RUN_STAMP, df_no=df_no_physics, df_phys=df_physics)
else:
    print("Skipping grid search!")

print("run_gridsearch =", run_gridsearch)
print("hparams_exists =", hparams_exists(BASIN, MODE, RUN_LABEL))

In [ ]:
# Physics grid search merged into cell above

In [ ]:
try:
    if run_gridsearch:
        print("\n[INFO] Using best_params_df from the just-completed grid search (ignoring READ_STAMP).")
    else:
        print("\nLoading best hyperparams from CSV...")
        best_params_df = load_hparams(BASIN, MODE, RUN_LABEL, stamp=READ_STAMP)
except FileNotFoundError as e:
    raise SystemExit(f"[ERROR] {e}  (Set run_gridsearch=True to generate it.)")

best_no_phys = best_params_df.query("model_type == 'no_physics'").iloc[0].to_dict()
best_phys = best_params_df.query("model_type == 'physics'").iloc[0].to_dict()

best_no_physics_params = {}
j = 0
while j < len(hyperparam_names):
    name = hyperparam_names[j]
    if name == "output_dropout":
        best_no_physics_params[name] = float(best_no_phys[name])
        j += 1

    elif name == "seq_length":
        best_no_physics_params["seq_length"] = int(best_no_phys["seq_length"])
        j += 1

    elif name == "schedule_pairs":
        j += 1

    else:
        best_no_physics_params[name] = int(best_no_phys[name])
        j += 1

if "learning_rate" in best_no_phys and pd.notna(best_no_phys["learning_rate"]):
    best_no_physics_params["learning_rate"] = eval(str(best_no_phys["learning_rate"]))

elif "schedule_pairs" in best_no_phys and pd.notna(best_no_phys["schedule_pairs"]):
    sp = best_no_phys["schedule_pairs"]
    if isinstance(sp, str):
        sp = eval(sp)
    fractions, rates = sp
    best_no_physics_params["learning_rate"] = fractional_multi_lr(
        epochs=int(best_no_physics_params["epochs"]),
        fractions=list(fractions),
        lrs=list(rates))

else:
    best_no_physics_params["learning_rate"] = {0: 0.01, 30: 0.005, 40: 0.001}

best_physics_params = {}
j = 0
while j < len(hyperparam_names):
    name = hyperparam_names[j]
    if name == "output_dropout":
        best_physics_params[name] = float(best_phys[name])
        j += 1

    elif name == "seq_length":
        best_physics_params["seq_length"] = int(best_phys["seq_length"])
        j += 1

    elif name == "schedule_pairs":
        j += 1

    else:
        best_physics_params[name] = int(best_phys[name])
        j += 1

if "learning_rate" in best_phys and pd.notna(best_phys["learning_rate"]):
    best_physics_params["learning_rate"] = eval(str(best_phys["learning_rate"]))

elif "schedule_pairs" in best_phys and pd.notna(best_phys["schedule_pairs"]):
    sp = best_phys["schedule_pairs"]
    if isinstance(sp, str):
        sp = eval(sp)
    fractions, rates = sp
    best_physics_params["learning_rate"] = fractional_multi_lr(
        epochs=int(best_physics_params["epochs"]),
        fractions=list(fractions),
        lrs=list(rates))

else:
    best_physics_params["learning_rate"] = {0: 0.01, 30: 0.005, 40: 0.001}

print("Loaded best hyperparams from CSV:")
print("Best NO-PHYS:", best_no_physics_params)
print("Best PHYS:", best_physics_params)

### Re-run validation with best hyperparameters

In [ ]:
if not skip_best_model_validation and not use_cv_for_selection:
    lstmNoPhysicsValBest = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_no_physics_params,
        input_features=None,
        physics_informed=False,
        physics_data_file=None,
        hourly=False,
        extend_train_period=False,
        gpu=GPU_SETTING,
        # num_ensemble_members = NUM_ENSEMBLES_FINAL,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP)
    
    lstmNoPhysicsValBest.train()
    no_physics_val_csv, no_physics_val_metrics = lstmNoPhysicsValBest.results()
    no_physics_val_metrics

In [ ]:
if not skip_best_model_validation and not use_cv_for_selection:
    lstmPhysicsValBest = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_physics_params,
        input_features=features_with_physics,
        physics_informed=True,
        physics_data_file=path_to_physics_data,
        hourly=False,
        extend_train_period=False,
        gpu=GPU_SETTING,
        # num_ensemble_members = NUM_ENSEMBLES_FINAL,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP)
    
    lstmPhysicsValBest.train()
    physics_val_csv, physics_val_metrics = lstmPhysicsValBest.results()
    physics_val_metrics

In [ ]:
if use_cv_for_selection:
    pass  # No validation period in CV mode
elif not skip_best_model_validation:
    plot_timeseries_comparison(source=(no_physics_val_csv, physics_val_csv, path_to_physics_data), title="Warm Springs Basin Daily Model Comparison (Validation)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_daily_val_metrics.csv", ts_out="warm_springs_daily_val_combined_ts.csv", fig_out="warm_springs_daily_val_model_comparison.png", legend_font=20, axis_font=22)
else:
    combined_df_val = read_csv_artifact("warm_springs_daily_val_combined_ts.csv", kind="csv", period="validation", stamp = READ_STAMP, run_label = RUN_LABEL)
    plot_timeseries_comparison(source=combined_df_val, title="Warm Springs Basin Daily Model Comparison (Validation)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_daily_val_metrics.csv", ts_out="warm_springs_daily_val_combined_ts.csv", fig_out="warm_springs_daily_val_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if skip_best_model_validation:
    val_metrics = read_csv_artifact("warm_springs_daily_val_metrics.csv", kind="metrics", period="validation", index_col=0, stamp = READ_STAMP, run_label = RUN_LABEL)
    print(val_metrics)

### Re-run testing with best hyperparameters

In [ ]:
if not skip_best_model_test:
    lstmNoPhysicsExtBest = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_no_physics_params,
        input_features=None,
        physics_informed=False,
        physics_data_file=None,
        hourly=False,
        extend_train_period=True,  
        gpu=GPU_SETTING,
        # num_ensemble_members = NUM_ENSEMBLES_FINAL,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP)
    
    lstmNoPhysicsExtBest.train()
    no_physics_test_csv, no_physics_test_metrics = lstmNoPhysicsExtBest.results('test')
    no_physics_test_metrics

In [ ]:
if not skip_best_model_test:
    lstmPhysicsExtBest = UCB_trainer(
        path_to_csv_folder=path_to_csv,
        yaml_path=path_to_yaml,
        hyperparams=best_physics_params,
        input_features=features_with_physics,
        physics_informed=True,
        physics_data_file=path_to_physics_data,
        hourly=False,
        extend_train_period=True,
        gpu=GPU_SETTING,
        # num_ensemble_members = NUM_ENSEMBLES_FINAL,
        verbose=verbose,
        runs_parent=RUNS_PARENT,
        run_label=RUN_LABEL,
        run_stamp=RUN_STAMP)
    
    lstmPhysicsExtBest.train()
    physics_test_csv, physics_test_metrics = lstmPhysicsExtBest.results('test')
    physics_test_metrics

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Basin Daily Model Comparison (Test)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_model_comparison.png", legend_font=20, axis_font=22)
else:
    combined_df = read_csv_artifact("warm_springs_daily_test_combined_ts.csv", kind="csv", period="test", stamp = READ_STAMP, run_label = RUN_LABEL)
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Basin Daily Model Comparison (Test)", backend="mpl", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Basin Daily Model Comparison (Test)", backend="plotly", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_model_comparison.png", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Basin Daily Model Comparison (Test)", backend="plotly", metrics=["NSE", "PBIAS"], metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_model_comparison.png", legend_font=12, axis_font=22)

In [ ]:
test_metrics = read_csv_artifact("warm_springs_daily_test_metrics.csv", kind="metrics", period="test", index_col=0, stamp = READ_STAMP, run_label = RUN_LABEL)
print(test_metrics)

In [ ]:
if not skip_best_model_validation:
    write_paths("no_physics_val", lstmNoPhysicsValBest, filename = RUNS_FILE)
    write_paths("physics_val", lstmPhysicsValBest, filename = RUNS_FILE)

if not skip_best_model_test:
    write_paths("no_physics", lstmNoPhysicsExtBest, filename = RUNS_FILE)
    write_paths("physics", lstmPhysicsExtBest, filename = RUNS_FILE)
    archived_path = archive_runs_json(Path(RUNS_FILE), BASIN, MODE, RUN_LABEL, RUN_STAMP)

In [ ]:
end_time = datetime.utcnow()
print("\nEnd time:", end_time.strftime("%Y-%m-%d %H:%M:%S"))
print("Total time:", end_time - start_time)

##### Additional Plots

In [ ]:
if skip_best_model_validation:
    combined_df_val = read_csv_artifact("warm_springs_daily_val_combined_ts.csv", kind="csv", period="validation", stamp = READ_STAMP, run_label = RUN_LABEL)
if skip_best_model_test:
    combined_df = read_csv_artifact("warm_springs_daily_test_combined_ts.csv", kind="csv", period="test", stamp = READ_STAMP, run_label = RUN_LABEL)

In [ ]:
metric_list = ["NSE", "PBIAS"]

wettest_start_val = "2003-10-01"
wettest_end_val = "2004-09-30"
dryest_start_val = "2004-10-01"
dryest_end_val = "2005-09-30"
wettest_start_test = "2005-10-01"
wettest_end_test = "2006-09-30"
dryest_start_test = "2008-10-01"
dryest_end_test = "2009-09-30"

In [ ]:
if use_cv_for_selection:
    pass  # No validation period in CV mode
elif not skip_best_model_validation:
    plot_timeseries_comparison(source=(no_physics_val_csv, physics_val_csv, path_to_physics_data), title="Warm Springs Daily Validation Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_daily_val_metrics.csv", ts_out="warm_springs_daily_val_combined_ts.csv", fig_out="warm_springs_daily_val_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_val, title="Warm Springs Daily Validation Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_daily_val_metrics.csv", ts_out="warm_springs_daily_val_combined_ts.csv", fig_out="warm_springs_daily_val_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Daily Test Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_test_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Daily Test Timeseries", backend="mpl", metrics=metric_list, metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_test_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Daily Test Timeseries – Interactive", backend="plotly", metrics=metric_list, metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Daily Test Timeseries – Interactive", backend="plotly", metrics=metric_list, metrics_out="warm_springs_daily_test_metrics.csv", ts_out="warm_springs_daily_test_combined_ts.csv", fig_out="warm_springs_daily_test_model_comparison.png", legend_font=12, axis_font=22)

##### Wettest Year Performance

In [ ]:
if use_cv_for_selection:
    pass  # No validation period in CV mode
elif not skip_best_model_validation:
    plot_timeseries_comparison(source=(no_physics_val_csv, physics_val_csv, path_to_physics_data), title="Warm Springs Daily Wettest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_val, end_date=wettest_end_val, metrics_out="warm_springs_daily_val_wet_metrics.csv", ts_out="warm_springs_daily_val_wet_combined_ts.csv", fig_out="warm_springs_daily_val_wet_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_val, title="Warm Springs Daily Wettest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_val, end_date=wettest_end_val, metrics_out="warm_springs_daily_val_wet_metrics.csv", ts_out="warm_springs_daily_val_wet_combined_ts.csv", fig_out="warm_springs_daily_val_wet_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Daily Wettest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_daily_test_wet_metrics.csv", ts_out="warm_springs_daily_test_wet_combined_ts.csv", fig_out="warm_springs_daily_test_wet_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Daily Wettest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_daily_test_wet_metrics.csv", ts_out="warm_springs_daily_test_wet_combined_ts.csv", fig_out="warm_springs_daily_test_wet_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Daily Wettest Year Test Timeseries – Interactive", backend="plotly", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_daily_test_wet_metrics.csv", ts_out="warm_springs_daily_test_wet_combined_ts.csv", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Daily Wettest Year Test Timeseries – Interactive", backend="plotly", metrics=metric_list, start_date=wettest_start_test, end_date=wettest_end_test, metrics_out="warm_springs_daily_test_wet_metrics.csv", ts_out="warm_springs_daily_test_wet_combined_ts.csv", legend_font=12, axis_font=22)

##### Dryest Year Performance

In [ ]:
if use_cv_for_selection:
    pass  # No validation period in CV mode
elif not skip_best_model_validation:
    plot_timeseries_comparison(source=(no_physics_val_csv, physics_val_csv, path_to_physics_data), title="Warm Springs Daily Dryest Year Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_val, end_date=dryest_end_val, metrics_out="warm_springs_daily_val_dry_metrics.csv", ts_out="warm_springs_daily_val_dry_combined_ts.csv", fig_out="warm_springs_daily_val_dry_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df_val, title="Warm Springs Daily Dryest Year Validation Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_val, end_date=dryest_end_val, metrics_out="warm_springs_daily_val_dry_metrics.csv", ts_out="warm_springs_daily_val_dry_combined_ts.csv", fig_out="warm_springs_daily_val_dry_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Daily Driest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_daily_test_dry_metrics.csv", ts_out="warm_springs_daily_test_dry_combined_ts.csv", fig_out="warm_springs_daily_test_dry_model_comparison.png", legend_font=20, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Daily Driest Year Test Timeseries", backend="mpl", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_daily_test_dry_metrics.csv", ts_out="warm_springs_daily_test_dry_combined_ts.csv", fig_out="warm_springs_daily_test_dry_model_comparison.png", legend_font=20, axis_font=22)

In [ ]:
if not skip_best_model_test:
    plot_timeseries_comparison(source=(no_physics_test_csv, physics_test_csv, path_to_physics_data), title="Warm Springs Daily Driest Year Test Timeseries – Interactive", backend="plotly", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_daily_test_dry_metrics.csv", ts_out="warm_springs_daily_test_dry_combined_ts.csv", legend_font=12, axis_font=22)
else:
    plot_timeseries_comparison(source=combined_df, title="Warm Springs Daily Driest Year Test Timeseries – Interactive", backend="plotly", metrics=metric_list, start_date=dryest_start_test, end_date=dryest_end_test, metrics_out="warm_springs_daily_test_dry_metrics.csv", ts_out="warm_springs_daily_test_dry_combined_ts.csv", legend_font=12, axis_font=22)

##### Gridded Timeseries Plots - Validation & Testing

In [ ]:
if not skip_best_model_validation:
    ts_triptych_v3((no_physics_val_csv, physics_val_csv, path_to_physics_data),wet_start=wettest_start_val, wet_end=wettest_end_val, dry_start=dryest_start_val, dry_end=dryest_end_val, save_path="warm_springs_daily_TS_validation.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d-%b-%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title="Warm Springs Daily Validation Period Timeseries Across Models", main_title_font=14, main_title_y=0.99, main_title_pad=0.05, row_titles=("Full Validation period","Most-wet water-year","Most-dry water-year"))

else:
    ts_triptych_v3(combined_df_val, wet_start=wettest_start_val, wet_end=wettest_end_val, dry_start=dryest_start_val, dry_end=dryest_end_val, save_path="warm_springs_daily_TS_validation.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d‑%b‑%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title = "Warm Springs Daily Validation Timeseries Across Models", main_title_font=14, main_title_y = 0.99, main_title_pad = 0.05, row_titles=("Full validation period", "Most‑wet water‑year", "Most‑dry water‑year"))

In [ ]:
if not skip_best_model_test:
    ts_triptych_v3((no_physics_test_csv, physics_test_csv, path_to_physics_data), wet_start=wettest_start_test, wet_end=wettest_end_test, dry_start=dryest_start_test, dry_end=dryest_end_test, save_path="warm_springs_daily_TS_testing.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d-%b-%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title="Warm Springs Daily Test Period Timeseries Across Models", main_title_font=14, main_title_y=0.99, main_title_pad=0.05, row_titles=("Full Testing period","Most-wet water-year","Most-dry water-year"))
    
else:
    ts_triptych_v3(combined_df, wet_start=wettest_start_test, wet_end=wettest_end_test, dry_start=dryest_start_test, dry_end=dryest_end_test, save_path="warm_springs_daily_TS_testing.png", legend_font=12, legend_boxpad=0.5, axis_font=12, date_fmt="%d‑%b‑%Y", figsize=(10, 10), dpi=600, hspace=0.2, main_title = "Warm Springs Daily Test Period Timeseries Across Models", main_title_font=14, main_title_y = 0.99, main_title_pad = 0.05, row_titles=("Full Testing period", "Most‑wet water‑year", "Most‑dry water‑year"))

##### Gridded Scatter Plots - Testing

In [ ]:
if not skip_best_model_test:
    scatter_pngs = scatter_triptych_pngs_v3((no_physics_test_csv, physics_test_csv, path_to_physics_data), wet_start = wettest_start_test, wet_end = wettest_end_test, dry_start = dryest_start_test,  dry_end = dryest_end_test, out_dir = "warm_springs_daily_scatter", layout = "horizontal", square_side  = 4.5, legend_font  = 16, axis_font = 16, point_size = 28, top_pad = .90, suptitle_y = 1.04, dpi = 600, row_titles = ("Warm Springs Daily – Full test period", "Warm Springs Daily – Wettest water‑year", "Warm Springs Daily – Driest water‑year"), resolution = "daily")
    
else:
    scatter_pngs = scatter_triptych_pngs_v3(combined_df, wet_start = wettest_start_test, wet_end = wettest_end_test, dry_start = dryest_start_test,  dry_end = dryest_end_test, out_dir = "warm_springs_daily_scatter", layout = "horizontal", square_side  = 4.5, legend_font  = 16, axis_font = 16, point_size = 28, top_pad = .90, suptitle_y = 1.04, dpi = 600, row_titles = ("Warm Springs Daily – Full test period", "Warm Springs Daily – Wettest water‑year", "Warm Springs Daily – Driest water‑year"), resolution = "daily")